In [1]:
import polars as pl

In [2]:
# load data
df = pl.read_ndjson("data/internal.jsonl")

In [3]:
df.columns

['user_agent.category',
 'source.ip',
 '@timestamp',
 'agent.hostname',
 'user_agent.version',
 'user_agent.original',
 'http.request.method',
 'agent.name',
 'source.registered_domain',
 'input.type',
 'event.blocked',
 'correlation',
 'destination.geo',
 '@version',
 'http.response.status_code',
 'attack',
 'agent.type',
 'event.created',
 'url.path',
 'source.domain',
 'source.geo',
 'ecs.version',
 'agent.id',
 'user_agent.os.platform',
 'log.file.path',
 'url.controller',
 'url.filetype',
 'metrics',
 'event.dataset',
 'http.version',
 'http.response.status',
 'source.as',
 'destination.as',
 'url.original',
 'agent.ephemeral_id',
 'agent.version',
 'user_agent.name',
 'user_agent.description',
 'service.group.id',
 'organization.name',
 'tags',
 'service.name',
 'event.module',
 'service.type',
 'user_agent.os.device.name',
 'destination.ip',
 'service.group.name',
 'log.offset',
 'http.response.bytes',
 'source.network']

In [4]:
print(df.head(3))

shape: (3, 50)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ user_agen ┆ source.ip ┆ @timestam ┆ agent.hos ┆ … ┆ service.g ┆ log.offse ┆ http.resp ┆ source.n │
│ t.categor ┆ ---       ┆ p         ┆ tname     ┆   ┆ roup.name ┆ t         ┆ onse.byte ┆ etwork   │
│ y         ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ s         ┆ ---      │
│ ---       ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ i64       ┆ ---       ┆ struct[6 │
│ str       ┆           ┆           ┆           ┆   ┆           ┆           ┆ str       ┆ ]        │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ SCRIPTING ┆ 93.94.228 ┆ 2020-12-2 ┆ gitlab    ┆ … ┆ default   ┆ 21297150  ┆ 4304      ┆ null     │
│           ┆ .141      ┆ 1T09:29:5 ┆           ┆   ┆           ┆           ┆           ┆          │
│           ┆           ┆ 2.000Z    ┆           ┆   ┆           ┆           

In [5]:
df["attack"].describe()

statistic,value
str,f64
"""count""",691303.0
"""null_count""",202.0


In [6]:
null_attack_rows = df.filter(
    pl.col("attack").is_null()
)
null_attack_rows.unique("source.ip").shape[0]
# null_attack_rows.get_column("source.ip").value_counts(sort=True).head(5)


99

In [7]:
df.select([
    pl.col(c).is_null().sum().alias(c)
    for c in df.columns
])

user_agent.category,source.ip,@timestamp,agent.hostname,user_agent.version,user_agent.original,http.request.method,agent.name,source.registered_domain,input.type,event.blocked,correlation,destination.geo,@version,http.response.status_code,attack,agent.type,event.created,url.path,source.domain,source.geo,ecs.version,agent.id,user_agent.os.platform,log.file.path,url.controller,url.filetype,metrics,event.dataset,http.version,http.response.status,source.as,destination.as,url.original,agent.ephemeral_id,agent.version,user_agent.name,user_agent.description,service.group.id,organization.name,tags,service.name,event.module,service.type,user_agent.os.device.name,destination.ip,service.group.name,log.offset,http.response.bytes,source.network
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
226,0,0,0,206771,226,0,6873,100231,0,0,0,6873,0,0,202,0,0,0,48114,0,0,0,226,0,0,0,0,0,0,0,0,6873,0,0,0,226,15050,0,0,0,0,0,0,226,6873,0,0,0,458575


In [8]:
df.shape

(691505, 50)

In [9]:
unique_counts = df.select([
    pl.col(c).n_unique().alias(c)
    for c in df.columns
])
unique_counts.to_pandas()

,user_agent.category,source.ip,@timestamp,agent.hostname,user_agent.version,user_agent.original,http.request.method,agent.name,source.registered_domain,input.type,...,tags,service.name,event.module,service.type,user_agent.os.device.name,destination.ip,service.group.name,log.offset,http.response.bytes,source.network
0,3,617,387008,2,59,175,7,2,97,1,...,7,3,1,3,7,2,1,687538,4199,28


In [10]:
cols_with_one_unique = (
    unique_counts.unpivot()
    .filter(pl.col("value") == 1)
    .select("variable")
    .to_series()
    .to_list()
)

print(cols_with_one_unique)

['input.type', 'event.blocked', '@version', 'agent.type', 'event.dataset', 'service.group.id', 'organization.name', 'event.module', 'service.group.name']


In [11]:
df_cleaned = df.drop(cols_with_one_unique)
df_cleaned = df_cleaned.drop("metrics")
df_cleaned = df_cleaned.drop("attack")
df_cleaned = df_cleaned.drop("log.file.path")
df_cleaned = df_cleaned.drop("log.offset")
df_cleaned.shape

(691505, 37)

In [12]:
df_cleaned["@timestamp"][0]

'2020-12-21T09:29:52.000Z'

In [13]:
df_cleaned = df_cleaned.with_columns(
    pl.col("@timestamp").str.to_datetime(time_zone="UTC").alias("timestamp")
).drop("@timestamp")

In [14]:
useful_cols = [
    "@timestamp",                  # for time-based features
    "source.ip",                  # key identifier for behavioral aggregation
    "source.registered_domain",   # contextual info
    "source.as",                  # ASN for network context
    "source.geo",                 # localization info
    "destination.ip",             # optional, useful for target analysis
    "http.request.method",        # method of request
    "http.response.status_code",  # for success/failure ratios
    "http.response.bytes",        # request size, can indicate scanning or attacks
    "url.path",                   # endpoint / path entropy
    "url.filetype",               # optional for content-specific analysis
    "user_agent.name",            # agent type
    "user_agent.version",         # optional, more granular agent info
    "user_agent.os.platform",     # device OS / platform
    "user_agent.os.device.name",  # device type
    "user_agent.category",        # e.g., browser, bot, API
    "agent.name",                 # collector / log agent
    "agent.id",                   # collector id if multiple agents
    "tags",                       # any existing labeling/tags
    "correlation"                 # optional for linked events
]


In [15]:
df_cleaned = df.select([pl.col(c) for c in useful_cols])

In [16]:
df_cleaned = df_cleaned.rename({"@timestamp": "timestamp"})

In [17]:
df_cleaned.head(3)

timestamp,source.ip,source.registered_domain,source.as,source.geo,destination.ip,http.request.method,http.response.status_code,http.response.bytes,url.path,url.filetype,user_agent.name,user_agent.version,user_agent.os.platform,user_agent.os.device.name,user_agent.category,agent.name,agent.id,tags,correlation
str,str,str,struct[2],struct[4],str,str,str,str,str,str,str,str,str,str,str,str,str,list[str],struct[1]
"""2020-12-21T09:29:52.000Z""","""93.94.228.141""","""egis.cyso.net""","{""Cyso Group B.V."",25151}","{{52.6318,4.7409},""Alkmaar"",""Netherlands"",""NL""}","""116.203.157.62""","""GET""","""200""","""4304""","""/v2/securely/front-end/securel…","""develop""","""Go-http-client""","""1""","""Other""","""Other""","""SCRIPTING""","""gitlab""","""ad99edda-a068-4794-b2c5-201664…","[""beats_input_codec_plain_applied"", ""exclusion-exclude-on-unknown-user-agent-is-used-to-perform-request--1580584312"", ""file_backup""]","{[""customer-onprem""]}"
"""2020-12-21T09:29:52.000Z""","""93.94.228.141""","""egis.cyso.net""","{""Cyso Group B.V."",25151}","{{52.6318,4.7409},""Alkmaar"",""Netherlands"",""NL""}","""116.203.157.62""","""GET""","""200""","""4929""","""/v2/logstash/logstash-config/m…","""latest""","""Go-http-client""","""1""","""Other""","""Other""","""SCRIPTING""","""gitlab""","""ad99edda-a068-4794-b2c5-201664…","[""beats_input_codec_plain_applied"", ""exclusion-exclude-on-unknown-user-agent-is-used-to-perform-request--1580584312"", ""file_backup""]","{[""customer-onprem""]}"
"""2020-12-21T09:29:53.000Z""","""84.207.227.190""",null,"{""euNetworks GmbH"",13237}","{{52.3759,4.8975},""Amsterdam"",""Netherlands"",""NL""}","""116.203.157.62""","""GET""","""200""","""4929""","""/v2/logstash/logstash-config/m…","""6-add-support-for-palo-alto-fr…","""Go-http-client""","""1""","""Other""","""Other""","""SCRIPTING""","""gitlab""","""ad99edda-a068-4794-b2c5-201664…","[""beats_input_codec_plain_applied"", ""exclusion-exclude-on-unknown-user-agent-is-used-to-perform-request--1580584312"", ""file_backup""]","{[""customer-onprem""]}"


In [18]:
df_cleaned.select([
    pl.col(c).is_null().sum().alias(c)
    for c in df_cleaned.columns
]).unpivot().filter(
    pl.col("value") > 0
).rename({
    "variable": "column",
    "value": "null_count"
}).sort("null_count", descending=True)


column,null_count
str,u32
"""user_agent.version""",206771
"""source.registered_domain""",100231
"""destination.ip""",6873
"""agent.name""",6873
"""user_agent.name""",226
"""user_agent.os.platform""",226
"""user_agent.os.device.name""",226
"""user_agent.category""",226


In [19]:
df_cleaned = df_cleaned.with_columns(
    pl.col("timestamp").str.strptime(pl.Datetime, format="%Y-%m-%dT%H:%M:%S.%3fZ", strict=False)
)

In [20]:
df_cleaned = df_cleaned.with_columns(
    pl.col("http.response.bytes")
    .cast(pl.Int64, strict=False)
    .fill_null(0)                 
)
print(df_cleaned.get_column("http.response.bytes").dtype)

Int64


In [24]:
df_cleaned = df_cleaned.with_columns(
    pl.col("source.geo").struct.field("country_iso_code").fill_null("unknown").alias("geo_code")
)

In [28]:
df_cleaned.write_parquet("data/cleaned.parquet")